# Figures

This notebook reads the CSVs produced by the experiment scripts in
`experiments/` and generates the figures used in the paper.

Run the experiment scripts first:

```bash
python -m experiments.exp1_rotation
python -m experiments.exp2_translation
python -m experiments.exp3_homothety
```

This produces `results/exp1_rotation.csv`, `results/exp2_translation.csv`
and `results/exp3_homothety.csv`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make `src` importable from the notebook.
REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RESULTS = REPO_ROOT / 'results'
FIGURES = REPO_ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)

METHOD_COLORS = {
    'TO':  'lightblue',
    'LFT': 'steelblue',
    'TL':  'orange',
    'ALR': 'lightgreen',
}
METHOD_ORDER = ['TO', 'LFT', 'TL', 'ALR']

## Experiment 1 — Rotation

Mean and std of MSE per (theta, sigma).

In [ ]:
df1 = pd.read_csv(RESULTS / 'exp1_rotation.csv')

theta_map = {
    (5/6)*np.pi: '5/6\u03c0',
    (3/4)*np.pi: '3/4\u03c0',
    (2/3)*np.pi: '2/3\u03c0',
    ((1/2) + (1/20))*np.pi: '11/20\u03c0',
    ((1/2) - (1/20))*np.pi: '9/20\u03c0',
    (1/3)*np.pi: '1/3\u03c0',
    (1/4)*np.pi: '1/4\u03c0',
    (1/6)*np.pi: '1/6\u03c0',
}

summary = df1.groupby(['theta', 'noise']).agg(
    mean_TO=('mse_TO', 'mean'),  std_TO=('mse_TO', 'std'),
    mean_ALR=('mse_ALR','mean'), std_ALR=('mse_ALR','std'),
    mean_LFT=('mse_LFT','mean'), std_LFT=('mse_LFT','std'),
    mean_TL=('mse_TL', 'mean'),  std_TL=('mse_TL', 'std'),
).reset_index()
summary['theta_label'] = summary['theta'].apply(
    lambda x: theta_map.get(x, f'{x/np.pi:.2f}\u03c0')
)
summary.round(4).head(20)

## Experiment 2 — Translation

In [ ]:
df2 = pd.read_csv(RESULTS / 'exp2_translation.csv')

norms = sorted(df2['norm'].unique())
fig, axs = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Translation experiment - MSE by method', fontsize=14)

for ax, norm in zip(axs.flat, norms):
    sub = df2[df2['norm'] == norm]
    data = [sub[f'mse_{m}'].values for m in METHOD_ORDER]
    bp = ax.boxplot(data, positions=range(1, 5), widths=0.6, patch_artist=True)
    for patch, m in zip(bp['boxes'], METHOD_ORDER):
        patch.set_facecolor(METHOD_COLORS[m])
    ax.set_xticks(range(1, 5))
    ax.set_xticklabels(METHOD_ORDER)
    ax.set_title(f'$||v||={norm}$')
    ax.set_ylabel('MSE')
    ax.grid(axis='y')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(FIGURES / 'exp2_translation_boxplots.pdf', bbox_inches='tight')
plt.show()

## Experiment 3 — Homothety

In [ ]:
df3 = pd.read_csv(RESULTS / 'exp3_homothety.csv')

lambdas = sorted(df3['lambda'].unique())
fig, axs = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Homothety experiment - MSE by method', fontsize=14)

for ax, lam in zip(axs.flat, lambdas):
    sub = df3[df3['lambda'] == lam]
    data = [sub[f'mse_{m}'].values for m in METHOD_ORDER]
    bp = ax.boxplot(data, positions=range(1, 5), widths=0.6, patch_artist=True)
    for patch, m in zip(bp['boxes'], METHOD_ORDER):
        patch.set_facecolor(METHOD_COLORS[m])
    ax.set_xticks(range(1, 5))
    ax.set_xticklabels(METHOD_ORDER)
    ax.set_title(f'$\\lambda={lam}$')
    ax.set_ylabel('MSE')
    ax.grid(axis='y')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(FIGURES / 'exp3_homothety_boxplots.pdf', bbox_inches='tight')
plt.show()